# Multimodal pair verification (public data)

Train a regularized pair verifier on matching pairs, harder negatives selected by cosine similarity, and random negatives. Compare against cosine similarity across every held-out pair. Report AUROC, average precision and image-to-text Recall@5. Use the official 200 training pairs and 80 test pairs sampled from Radiology_mini with the cached frozen CLIP encoder. All three fits share one test gallery. The task links an image to its source caption.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'crossmodal-pair-audit-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0', 'torch==2.14.1', 'datasets==5.2.0', 'transformers==5.20.2', 'Pillow==12.3.0']],check=True)

In [ ]:
(root / 'prepare.py').write_text('from __future__ import annotations\n\nimport csv\nimport json\nimport os\nimport random\nfrom itertools import islice\nfrom pathlib import Path\n\nos.environ.setdefault(\n    "HF_HOME", str(Path(__file__).resolve().parent / "data" / "hf_cache")\n)\nos.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom datasets import load_dataset\nfrom PIL import ImageFilter\nfrom transformers import AutoProcessor, CLIPModel\n\nROOT = Path(__file__).resolve().parent\n\nDATA_DIR = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nEMBEDDING_PATH = DATA_DIR / "roco_clip_embeddings.npz"\nMODEL_NAME = "openai/clip-vit-base-patch32"\nTRAIN_COUNT = 200\nTEST_COUNT = 80\nBATCH_SIZE = 16\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n\ndef collect(split, count):\n    stream = load_dataset(\n        "unsloth/Radiology_mini",\n        split=split,\n        streaming=True,\n        revision="12468bc1f3cecf1056c290182a0e88ce506049f9",\n    )\n    rows = []\n    for example in islice(stream, count):\n        rows.append(\n            {\n                "image": example["image"].convert("RGB").copy(),\n                "image_id": example["image_id"],\n                "caption": example["caption"],\n            }\n        )\n    if len(rows) != count:\n        raise RuntimeError(f"Expected {count} {split} rows, received {len(rows)}")\n    return rows\n\n\ndef dropped_caption(caption, seed):\n    words = caption.split()\n    if len(words) <= 3:\n        return caption\n    rng = random.Random(seed)\n    kept = [word for word in words if rng.random() > 0.35]\n    return " ".join(kept if len(kept) >= 3 else words[:3])\n\n\ndef embed_images(model, processor, images):\n    outputs = []\n    for start in range(0, len(images), BATCH_SIZE):\n        batch = processor(\n            images=images[start : start + BATCH_SIZE], return_tensors="pt"\n        )\n        batch = {key: value.to(DEVICE) for key, value in batch.items()}\n        with torch.no_grad():\n            features = model.get_image_features(**batch)\n            if hasattr(features, "pooler_output"):\n                features = features.pooler_output\n        outputs.append(F.normalize(features, dim=1).cpu())\n        print(\n            json.dumps(\n                {\n                    "embedded_images": min(start + BATCH_SIZE, len(images)),\n                    "total": len(images),\n                }\n            ),\n            flush=True,\n        )\n    return torch.cat(outputs).numpy().astype(np.float32)\n\n\ndef embed_text(model, processor, captions):\n    outputs = []\n    for start in range(0, len(captions), BATCH_SIZE * 2):\n        batch = processor(\n            text=captions[start : start + BATCH_SIZE * 2],\n            padding=True,\n            truncation=True,\n            return_tensors="pt",\n        )\n        batch = {key: value.to(DEVICE) for key, value in batch.items()}\n        with torch.no_grad():\n            features = model.get_text_features(**batch)\n            if hasattr(features, "pooler_output"):\n                features = features.pooler_output\n        outputs.append(F.normalize(features, dim=1).cpu())\n    return torch.cat(outputs).numpy().astype(np.float32)\n\n\ndef prepare_embeddings():\n    if EMBEDDING_PATH.exists():\n        return dict(np.load(EMBEDDING_PATH))\n    DATA_DIR.mkdir(parents=True, exist_ok=True)\n    train_rows = collect("train", TRAIN_COUNT)\n    test_rows = collect("test", TEST_COUNT)\n    with (DATA_DIR / "sample_manifest.csv").open(\n        "w", newline="", encoding="utf-8"\n    ) as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["split", "image_id", "caption"])\n        writer.writerows(\n            [["train", row["image_id"], row["caption"]] for row in train_rows]\n        )\n        writer.writerows(\n            [["test", row["image_id"], row["caption"]] for row in test_rows]\n        )\n    processor = AutoProcessor.from_pretrained(\n        MODEL_NAME, revision="3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"\n    )\n    model = (\n        CLIPModel.from_pretrained(\n            MODEL_NAME, revision="3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"\n        )\n        .to(DEVICE)\n        .eval()\n    )\n    train_images = [row["image"] for row in train_rows]\n    test_images = [row["image"] for row in test_rows]\n    train_captions = [row["caption"] for row in train_rows]\n    test_captions = [row["caption"] for row in test_rows]\n    arrays = {\n        "train_image": embed_images(model, processor, train_images),\n        "train_text": embed_text(model, processor, train_captions),\n        "test_image": embed_images(model, processor, test_images),\n        "test_text": embed_text(model, processor, test_captions),\n        "test_blur_image": embed_images(\n            model,\n            processor,\n            [image.filter(ImageFilter.GaussianBlur(3.0)) for image in test_images],\n        ),\n        "test_dropout_text": embed_text(\n            model,\n            processor,\n            [\n                dropped_caption(caption, 9000 + index)\n                for index, caption in enumerate(test_captions)\n            ],\n        ),\n    }\n    np.savez_compressed(EMBEDDING_PATH, **arrays)\n    return arrays\n\n\nif __name__ == "__main__":\n    prepare_embeddings()\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\ndef dataset(seed):\n    with np.load(DATA / "roco_clip_embeddings.npz") as a:\n        return tuple(\n            a[k] for k in ("train_image", "train_text", "test_image", "test_text")\n        )\n\n\ndef unit(x):\n    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-9)\n\n\ndef pair_features(a, b):\n    a, b = unit(a), unit(b)\n    return np.c_[np.sum(a * b, 1), np.abs(a - b), a * b]\n\n\ndef run(seed):\n    a, b, u, v = dataset(seed)\n    rng = np.random.default_rng(seed)\n    ids = np.arange(len(a))\n    np.sum(unit(a) * unit(b), axis=1)\n    cos = unit(a) @ unit(b).T\n    np.fill_diagonal(cos, -np.inf)\n    hard = np.argsort(-cos, axis=1)[:, :5]\n    neg = np.column_stack(\n        [\n            hard[:, 0],\n            hard[:, 2],\n            hard[:, 4],\n            (ids + rng.integers(1, len(a), len(a))) % len(a),\n        ]\n    )\n    n = neg.shape[1]\n    x = np.vstack(\n        [pair_features(a, b), pair_features(np.repeat(a, n, axis=0), b[neg.ravel()])]\n    )\n    y = np.r_[np.ones(len(a)), np.zeros(n * len(a))]\n    model = make_pipeline(\n        StandardScaler(),\n        LogisticRegression(C=0.005, class_weight="balanced", max_iter=400),\n    ).fit(x, y)\n    save(model, f"pair-model-{seed}")\n    reference = np.eye(len(u), dtype=bool).ravel()\n    paired = pair_features(np.repeat(u, len(v), axis=0), np.tile(v, (len(u), 1)))\n    out = {}\n    for method, s in [\n        ("cosine", paired[:, 0]),\n        ("learned", model.decision_function(paired)),\n    ]:\n        matrix = s.reshape(len(u), len(v))\n        out[method] = {\n            **scores(reference, s),\n            "recall_at_5": float(\n                np.mean(\n                    np.any(\n                        np.argsort(-matrix, axis=1)[:, :5]\n                        == np.arange(len(u))[:, None],\n                        axis=1,\n                    )\n                )\n            ),\n        }\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())